## 🤔 What Are Tools?

**Tools** in LangChain are interfaces that allow LLMs (Large Language Models) to interact with the external world.

Think of it this way:
- **LLMs alone**: Can only generate text based on their training data (knowledge cutoff)
- **LLMs with Tools**: Can perform actions, fetch real-time data, and interact with external systems

### Real-World Analogy
Imagine you're a brilliant person locked in a room with no access to the outside world:
- You can think and reason (like an LLM)
- But you can't check the weather, search the internet, or calculate complex math

**Tools are like giving you a phone, calculator, and internet connection!**

## 🎯 Why Do We Need Tools?

### 1. **Overcome Knowledge Cutoff**
- LLMs are trained on data up to a certain date
- Tools allow access to current information (weather, news, stock prices)

### 2. **Perform Actions**
- Send emails
- Make API calls
- Update databases
- Control IoT devices

### 3. **Complex Computations**
- Mathematical calculations
- Data analysis
- File operations

### 4. **Retrieval of Specific Information**
- Search databases
- Query APIs
- Web scraping

### 5. **Enhanced Reliability**
- For tasks requiring precision (like math), tools are more reliable than LLM generation

## 📦 Setup and Installation

In [1]:
# TODO: Install required packages
# pip install langchain langchain-openai langchain-community

# Import necessary libraries
import os
from langchain_core.tools import tool, StructuredTool
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

d:\Mentoring\learwithsarvesh\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# TODO: Import necessary libraries
# - tool, StructuredTool from langchain_core.tools
# - ChatOpenAI from langchain_openai
# - ChatPromptTemplate from langchain_core.prompts


import os
from dotenv import load_dotenv, find_dotenv

# Find and load the .env file from the current directory or parent directories
load_dotenv(find_dotenv(), override=True)

# Verify the API key is loaded (show first 10 chars only for security)
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"✅ API Key loaded: {api_key[:10]}...")
else:
    print("❌ API Key not found! Check your .env file")



✅ API Key loaded: sk-proj-3n...


In [ ]:
# TODO: Initialize the LLM
# - Use ChatOpenAI with model="gpt-4o-mini" and temperature=0

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 🔨 Creating Your First Custom Tool

Let's create a simple tool that calculates the square of a number.

### Method 1: Using the @tool Decorator (Recommended)

In [5]:
# TODO: Create a calculate_square tool using @tool decorator
# - Function should take a number as string input
# - Convert to float, calculate square
# - Return result as string
# - Include proper docstring

@tool
def calculate_square(number: str) -> str:
    """Calculates the square of a given number.
    
    Args:
        number: The number to square (as string)
    
    Returns:
        The square of the number
    """
    try:
        num = float(number)
        result = num ** 2
        return f"The square of {num} is {result}"
    except ValueError:
        return "Invalid input. Please provide a valid number."

# Test the tool
print(calculate_square.name)
print(calculate_square.description)
print(calculate_square.invoke("5"))    


calculate_square
Calculates the square of a given number.

    Args:
        number: The number to square (as string)

    Returns:
        The square of the number
The square of 5.0 is 25.0


### Method 2: Creating a Function and Wrapping with StructuredTool

This method is useful when you want to create a regular Python function first, then convert it to a tool.

In [6]:
# TODO: Create a multiply_numbers function
# - Create regular Python function (no decorator)
# - Wrap with StructuredTool.from_function()
# - Function should accept two numbers separated by comma (e.g., "5,3")
# - Return the product
def multiply_numbers(input_str: str) -> str:
    """ Multiply two numbers togeather """
    try:
        nums = input_str.split(",")
        if len(nums) != 2:
            return "Please provide exactly two numbers seperated by comma"        
        
        num1 = float(nums[0].strip())
        num2 = float(nums[1].strip())
        result = num1 * num2
        return f"{num1} × {num2} = {result}"
    except Exception as e:
        return f"Error: {str(e)}"

# Create the tool using StructuredTool.from_function (LangChain 1.0+ compatible)
multiply_tool = StructuredTool.from_function(
    func=multiply_numbers,
    name="Multiply",
    description="Useful for multiplying two numbers together. Input should be two numbers separated by comma, like '5,3'"
)

# Test it
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.invoke("6,7"))


Multiply
Useful for multiplying two numbers together. Input should be two numbers separated by comma, like '5,3'
6.0 × 7.0 = 42.0


## 🎨 More Practical Examples

Let's create more realistic tools that you might use in production.

### Example 1: Current Date/Time Tool

In [7]:
# TODO: Create a get_current_datetime tool
# - Import datetime
# - Create tool that returns current date/time
# - Support different formats: 'full', 'date', 'time'

from datetime import datetime

@tool
def get_current_datetime(format_type: str = "full") -> str:
    """Get the current date and time.
    
    Args:
        format_type: Type of format - 'full', 'date', or 'time'
    
    Returns:
        Current date/time in the requested format
    """
    now = datetime.now()
    
    if format_type == "date":
        return now.strftime("%Y-%m-%d")
    elif format_type == "time":
        return now.strftime("%H:%M:%S")
    else:
        return now.strftime("%Y-%m-%d %H:%M:%S")

# Test it
print(get_current_datetime.invoke("full"))
print(get_current_datetime.invoke("date"))
print(get_current_datetime.invoke("time"))


2025-12-18 12:15:20
2025-12-18
12:15:20


### Example 2: Text Analysis Tool

In [8]:
# TODO: Create an analyze_text tool
# - Count words, characters, sentences
# - Calculate average word length
# - Return formatted statistics

@tool
def analyze_text(text: str) -> str:
    """Analyze text and return statistics about it.
    
    Args:
        text: The text to analyze
    
    Returns:
        Statistics about the text including word count, character count, etc.
    """
    words = text.split()
    word_count = len(words)
    char_count = len(text)
    char_count_no_spaces = len(text.replace(" ", ""))
    sentence_count = text.count('.') + text.count('!') + text.count('?')
    
    analysis = f"""
    Text Analysis:
    - Word Count: {word_count}
    - Character Count (with spaces): {char_count}
    - Character Count (without spaces): {char_count_no_spaces}
    - Sentence Count: {sentence_count}
    - Average Word Length: {char_count_no_spaces/word_count if word_count > 0 else 0:.2f}
    """
    return analysis

# Test it
sample_text = "LangChain is amazing! It helps build AI applications. Tools are powerful."
print(analyze_text.invoke(sample_text))


    Text Analysis:
    - Word Count: 11
    - Character Count (with spaces): 73
    - Character Count (without spaces): 63
    - Sentence Count: 3
    - Average Word Length: 5.73
    


### Example 3: Temperature Converter Tool

In [9]:
# TODO: Create a convert_temperature tool
# - Convert between Celsius and Fahrenheit
# - Accept input like '25C' or '77F'
# - Return converted temperature
@tool
def convert_temperature(temp_input: str) -> str:
    """Convert temperature between Celsius and Fahrenheit.
    
    Args:
        temp_input: Temperature value followed by unit, e.g., '25C' or '77F'
    
    Returns:
        Converted temperature
    """
    try:
        # Parse input
        temp_input = temp_input.strip().upper()
        
        if temp_input.endswith('C'):
            celsius = float(temp_input[:-1])
            fahrenheit = (celsius * 9/5) + 32
            return f"{celsius}°C = {fahrenheit:.2f}°F"
        elif temp_input.endswith('F'):
            fahrenheit = float(temp_input[:-1])
            celsius = (fahrenheit - 32) * 5/9
            return f"{fahrenheit}°F = {celsius:.2f}°C"
        else:
            return "Please specify temperature with C or F, e.g., '25C' or '77F'"
    except Exception as e:
        return f"Error: {str(e)}"

# Test it
print(convert_temperature.invoke("25C"))
print(convert_temperature.invoke("77F"))
print(convert_temperature.invoke("0C"))

25.0°C = 77.00°F
77.0°F = 25.00°C
0.0°C = 32.00°F


## 🤖 Using Tools with Agents

Now let's see how to use these tools with an AI agent. The agent will automatically decide when and how to use the tools.

In [10]:
# TODO: Create a list of all tools
# - Add all created tools to a list
# - Print tool names and descriptions
# Create a list of tools
tools = [
    calculate_square,
    multiply_tool,
    get_current_datetime,
    analyze_text,
    convert_temperature
]

# Print available tools
print("Available Tools:")
for t in tools:  # Changed 'tool' to 't' to avoid shadowing the decorator
    print(f"- {t.name}: {t.description}")

Available Tools:
- calculate_square: Calculates the square of a given number.

    Args:
        number: The number to square (as string)

    Returns:
        The square of the number
- Multiply: Useful for multiplying two numbers together. Input should be two numbers separated by comma, like '5,3'
- get_current_datetime: Get the current date and time.

    Args:
        format_type: Type of format - 'full', 'date', or 'time'

    Returns:
        Current date/time in the requested format
- analyze_text: Analyze text and return statistics about it.

    Args:
        text: The text to analyze

    Returns:
        Statistics about the text including word count, character count, etc.
- convert_temperature: Convert temperature between Celsius and Fahrenheit.

    Args:
        temp_input: Temperature value followed by unit, e.g., '25C' or '77F'

    Returns:
        Converted temperature


In [11]:
# TODO: Bind tools to LLM and create run_agent function
# - Use llm.bind_tools(tools)
# - Create a helper function to:
#   1. Invoke LLM with user input
#   2. Check for tool calls
#   3. Execute tools if needed
#   4. Return results

# Bind tools to the LLM (LangChain 1.0.8 approach)
# This allows the LLM to automatically call tools when needed
llm_with_tools = llm.bind_tools(tools)

# Create a simple wrapper for easier invocation
def run_agent(user_input: str):
    """Helper function to run the agent with tool calling"""
    messages = [{"role": "user", "content": user_input}]
    response = llm_with_tools.invoke(messages)
    
    # If the LLM wants to call a tool
    if response.tool_calls:
        print(f"\n🔧 Agent is using tools: {[tc['name'] for tc in response.tool_calls]}")
        
        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            
            # Find and execute the tool
            for t in tools:
                if t.name == tool_name:
                    print(f"\n📍 Calling {tool_name} with args: {tool_args}")
                    result = t.invoke(tool_args)
                    print(f"✅ Result: {result}")
                    return result
    
    # If no tool call, return the response content
    return response.content

print("✅ Agent setup complete! Tools are bound to the LLM.")

✅ Agent setup complete! Tools are bound to the LLM.


### Test the Agent with Different Queries

In [12]:
# TODO: Query 1 - Mathematical operation
# Test: "What is the square of 12?"

print("Query: What is the square of 12?")

result = run_agent("What is the square of 12?")

print("\n" + "="*50)
print("FINAL ANSWER:", result)

Query: What is the square of 12?

🔧 Agent is using tools: ['calculate_square']

📍 Calling calculate_square with args: {'number': '12'}
✅ Result: The square of 12.0 is 144.0

FINAL ANSWER: The square of 12.0 is 144.0


In [ ]:
# TODO: Query 2 - Current date
# Test: "What's today's date?"


In [13]:
# Query 3: Temperature conversion
print("Query: Convert 100 degrees Fahrenheit to Celsius")
result = run_agent("Convert 100 degrees Fahrenheit to Celsius")

print("\n" + "="*50)
print("FINAL ANSWER:", result)

Query: Convert 100 degrees Fahrenheit to Celsius

🔧 Agent is using tools: ['convert_temperature']

📍 Calling convert_temperature with args: {'temp_input': '100F'}
✅ Result: 100.0°F = 37.78°C

FINAL ANSWER: 100.0°F = 37.78°C


In [14]:
# Query 4: Simple query (for demonstration)

print("Query: Calculate the square of 8")

result = run_agent("Calculate the square of 8")

print("\n" + "="*50)
print("FINAL ANSWER:", result)

Query: Calculate the square of 8

🔧 Agent is using tools: ['calculate_square']

📍 Calling calculate_square with args: {'number': '8'}
✅ Result: The square of 8.0 is 64.0

FINAL ANSWER: The square of 8.0 is 64.0


## 🌐 Real-World Tool Example: API Integration

Let's create a tool that fetches data from a real API.

In [16]:
# TODO: Create a get_random_joke tool
# - Import requests
# - Create tool that calls: https://official-joke-api.appspot.com/random_joke
# - Parse and return the joke

import requests
from langchain_core.tools import tool  # Re-import to ensure we have the decorator

@tool
def get_random_joke(category: str = "programming") -> str:
    """Fetch a random joke from an API.
    
    Args:
        category: Category of joke (not used in this simple example)
    
    Returns:
        A random joke
    """
    try:
        response = requests.get("https://official-joke-api.appspot.com/random_joke")
        if response.status_code == 200:
            joke_data = response.json()
            setup = joke_data.get('setup', '')
            punchline = joke_data.get('punchline', '')
            return f"{setup}\n{punchline}"
        else:
            return "Couldn't fetch a joke at the moment."
    except Exception as e:
        return f"Error fetching joke: {str(e)}"

# Test it
print(get_random_joke.invoke("programming"))

Did you hear about the submarine industry?
It really took a dive...


## 🏗️ Best Practices for Creating Tools

### 1. **Clear and Descriptive Names**
   - Use verb-noun format: `calculate_square`, `get_weather`
   - Make it obvious what the tool does

### 2. **Comprehensive Descriptions**
   - The description is crucial - the LLM uses it to decide when to use the tool
   - Include what it does, what input it expects, and what it returns

### 3. **Input Validation**
   - Always validate and handle errors
   - Provide helpful error messages

### 4. **Type Hints**
   - Use type hints for better code clarity
   - Tools accept string inputs by default

### 5. **Return Strings**
   - Tools should return strings that the LLM can understand
   - Format output clearly

## 🎓 Key Takeaways

1. **Tools extend LLM capabilities** - They allow LLMs to interact with the external world
2. **Two main methods** - `@tool` decorator (easier) or `StructuredTool` (more control)
3. **Description is critical** - The LLM uses descriptions to choose tools
4. **Agents orchestrate tools** - Agents decide which tools to use and when
5. **Real-world applications** - API calls, calculations, database queries, file operations

## 🚀 Next Steps

- Explore built-in LangChain tools (Wikipedia, Calculator, etc.)
- Create tools for your specific use case
- Build multi-tool agents for complex workflows
- Learn about tool error handling and retries
- Integrate with external APIs and services

## 💡 Exercise for Students

Try creating these tools on your own:

1. **Calculator Tool**: Create a tool that can perform basic arithmetic (+, -, *, /)
2. **String Reverser**: Tool that reverses a given string
3. **Password Generator**: Tool that generates a random password with specified length
4. **JSON Validator**: Tool that checks if a string is valid JSON
5. **Word Counter**: Tool that counts specific words in a text

**Bonus Challenge**: Create a tool that calls a public API of your choice!